# Validating behavioral transitions in mice

Run this notebook in `path to your RAW_DATA_DIR fold`, which also contains the data. It reads algorithm labels from `behavioral_scoring_validation video` and manual annotations from `expert 1` through `expert 5`, and saves the results in `analysis_outputs` under the same directory. Both analyses use transition events reconstructed from the original annotations. The existing English-language tables are available for video review.

I denotes immobility and S denotes struggling. The analysis begins at 120 s on the original video timeline. Analysis 1 uses the observation window specified for each mouse in `label.xlsx` to compare the onset and end of the same complete struggle bout across scorers. Analysis 2 reports the proportion of algorithm transitions that each expert matched within ±2 s.

In [ ]:
from itertools import combinations, product
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import FileLink, display
from matplotlib.colors import LinearSegmentedColormap

RAW_DATA_DIR = Path("path to your RAW_DATA_DIR fold")  # Input folder containing behavioral_scoring_validation video/, expert 1-5/, and label.xlsx.
ANALYSIS_DIR = RAW_DATA_DIR / "analysis_outputs"
ALGORITHM_START_S = 1.0
SAMPLE_RATE_HZ = 10
TRIM_SECONDS = 2.0
START_TIME_S = 120.0
MATCH_TOLERANCE_S = 2.0
FIGURE_DPI = 300
EXPERT_IDS = range(1, 6)
ALGORITHM_SCORER = "algorithm"
SCORERS = [ALGORITHM_SCORER, *(f"E{i}" for i in EXPERT_IDS)]
DIRECTIONS = ["I_to_S", "S_to_I"]
DIRECTION_LABELS = {"I_to_S": "I → S  Start struggling", "S_to_I": "S → I  Enter immobility"}

plt.rcParams.update({
    "font.family": "Arial",
    "text.color": "black",
    "axes.labelcolor": "black",
    "axes.titlecolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "legend.labelcolor": "black",
    "pdf.fonttype": 42,
    "pdf.use14corefonts": False,
    "text.usetex": False,
    "savefig.dpi": FIGURE_DPI,
})
ANIMAL_COLORS = ("#183f66", "#1b75ab", "#68a9ce", "#7a2344", "#b34038", "#e17b64")
BLUE_RED_CMAP = LinearSegmentedColormap.from_list(
    "match_rate_blue_red", ["#75afcb", "#fbfaf9", "#e78781"]
)
BLUE_RED_CMAP.set_bad("#f0f0f0")


def output_link(path):
    try:
        link_path = path.relative_to(Path.cwd())
    except ValueError:
        link_path = path
    display(FileLink(link_path.as_posix(), result_html_prefix="Saved: "))


def write_csv(frame, filename, *, index=False):
    frame.to_csv(ANALYSIS_DIR / filename, index=index, encoding="utf-8-sig", float_format="%.12g")


def save_figure(figure, filename):
    assert Path(filename).suffix == ".pdf"
    path = ANALYSIS_DIR / filename
    figure.savefig(path, format="pdf", dpi=FIGURE_DPI, bbox_inches="tight")
    plt.close(figure)
    output_link(path)



os.chdir(RAW_DATA_DIR)
ANALYSIS_DIR.mkdir(exist_ok=True)
print(f"Working directory: {Path.cwd()}")
print(f"Results: {ANALYSIS_DIR}")
print(f"Window starts at {START_TIME_S:g} s; tolerance ±{MATCH_TOLERANCE_S:g} s.")

## Reading annotations and reconstructing transitions

The algorithm uses a 10 Hz time grid. Its first bin is provisionally aligned to 1.0 s in the original video. Expert labels are mapped to the same grid using the frame rate of each video. Each bin takes the majority label, with ties resolved using the label at the center index. The original bidirectional transition filter and state reconstruction are applied before trimming 2 s from each end. The import step handles Excel-format annotations, animal ID aliases, and the known timestamp irregularities.

In [ ]:
import csv
from fractions import Fraction
from io import BytesIO
import json
import math
import shutil
import subprocess
from zipfile import is_zipfile

from openpyxl import load_workbook


def read_original_expert(path):
    # A file with a .csv suffix may contain an Excel workbook.
    if is_zipfile(path):
        workbook = load_workbook(BytesIO(path.read_bytes()), read_only=True, data_only=True)
        try:
            rows = list(workbook.active.values)
        finally:
            workbook.close()
        file_format = "Excel workbook"
    else:
        with path.open(encoding="utf-8-sig", newline="") as stream:
            rows = list(csv.reader(stream))
        file_format = "CSV"
    if tuple(rows[0]) != ("Time (s)", "Label"):
        raise ValueError(f"Unexpected expert header in {path}")
    times, labels = np.asarray(rows[1:], dtype=float).T
    if not np.isfinite(times).all() or not np.isin(labels, (0, 1)).all():
        raise ValueError(f"Invalid time or state label in {path}")
    return times, labels.astype(np.int8), file_format


def read_algorithm_labels(path):
    with path.open(encoding="utf-8-sig", newline="") as stream:
        header, values = list(csv.reader(stream))
    labels = np.asarray(values, dtype=float)
    if len(header) != len(labels) or not np.isin(labels, (0, 1)).all():
        raise ValueError(f"Malformed algorithm labels in {path}")
    return labels.astype(np.int8)


def read_video_clock(path, expert_reference_times):
    if shutil.which("ffprobe"):
        command = [
            "ffprobe", "-v", "error", "-select_streams", "v:0", "-show_entries",
            "stream=avg_frame_rate,nb_frames", "-of", "json", str(path)
        ]
        stream = json.loads(subprocess.check_output(command))["streams"][0]
        fps = Fraction(stream["avg_frame_rate"])
        if fps.denominator != 1:
            raise ValueError(f"Expected an integral video timebase rate: {path}")
        return int(fps), int(stream["nb_frames"]), "ffprobe"
    # Expert 1 timestamps provide the nominal time grid if ffprobe is absent.
    fps = int(round(1 / np.median(np.diff(expert_reference_times))))
    if not np.allclose(expert_reference_times, np.arange(len(expert_reference_times)) / fps,
                       rtol=0, atol=1e-5):
        raise ValueError(f"Cannot infer the time grid from Expert 1 for {path}")
    return fps, len(expert_reference_times), "Expert 1 time grid"



video_dirs = sorted(path for path in (RAW_DATA_DIR / "behavioral_scoring_validation video").iterdir() if path.is_dir())
if not video_dirs:
    raise ValueError(f"No video directories in {RAW_DATA_DIR}")
print(f"Found {len(video_dirs)} video sessions: " + ", ".join(p.name for p in video_dirs))

In [ ]:
def vote_expert_bins(states, first_frame, stop_frame, frames_per_sample):
    bins = states[first_frame:stop_frame].reshape(-1, frames_per_sample)
    votes = bins.sum(axis=1)
    voted = np.where(2 * votes == frames_per_sample,
                     bins[:, frames_per_sample // 2], 2 * votes > frames_per_sample)
    return voted.astype(np.int8)


def reconstruct_states(raw_states):
    "Use the original two-pass 21/19 sample rule and return 0=I, 1=S."
    edges = []
    for target in (1, 0):
        working_states = raw_states.copy()
        candidate_indices = np.flatnonzero(
            (raw_states[:-1] != target) & (raw_states[1:] == target)
        )
        for index in candidate_indices:
            if index < 20 or index > len(raw_states) - 20:
                continue
            if not np.all(working_states[index-20:index] == 1 - target):
                continue
            # This slice includes the old-state sample at index.
            if working_states[index:index+20].sum() == (19 if target == 1 else 1):
                edges.append((int(index), target))
                continue
            for following_index in range(index + 1, index + 20):
                if working_states[following_index] != target:
                    break
                working_states[following_index] = 1 - target

    edges.sort()
    retained_edges = [
        edge for position, edge in enumerate(edges)
        if position + 1 == len(edges) or edges[position + 1][1] != edge[1]
    ]
    if not retained_edges:
        if np.all(raw_states == raw_states[0]):
            return 1 - raw_states
        raise ValueError("A nonconstant trace has no accepted transitions")
    result = np.empty_like(raw_states)
    first_index, state = 0, retained_edges[0][1]
    for edge_index, target in retained_edges:
        result[first_index:edge_index] = state
        first_index, state = edge_index, 1 - target
    result[first_index:] = state
    return result


def inventory_from_reconstructed(states, start_sample, stop_sample):
    boundaries = np.flatnonzero(states[:-1] != states[1:]) + 1
    for sample_index in boundaries:
        if start_sample <= sample_index < stop_sample:
            yield int(sample_index), ("I_to_S" if states[sample_index] == 1 else "S_to_I")



EXPERT_ALIASES = {"M5": "M5"}  # Video and expert records use the same M1-M6 IDs; M5 retains the alias lookup.
KNOWN_TIMESTAMP_EXCEPTION = ("M3", 5)  # M3, expert 5: retain the original timestamp exception and frame-order handling.
trim_samples = int(round(TRIM_SECONDS * SAMPLE_RATE_HZ))
timeline_rows, event_rows = [], []

for directory in video_dirs:
    animal_id = directory.name
    videos = [path for path in directory.iterdir() if path.suffix.lower() in {".mp4", ".avi"}]
    if len(videos) != 1:
        raise ValueError(f"Expected one source video in {directory}, found {len(videos)}")
    expert_stem = EXPERT_ALIASES.get(animal_id, animal_id)
    experts = {
        expert_id: read_original_expert(RAW_DATA_DIR / f"expert {expert_id}" / f"{expert_stem}.csv")
        for expert_id in EXPERT_IDS
    }
    fps, frame_count, _ = read_video_clock(videos[0], experts[1][0])
    width = fps // SAMPLE_RATE_HZ
    if fps % SAMPLE_RATE_HZ:
        raise ValueError(f"Frames per sample must be integral for {animal_id}")
    algorithm = read_algorithm_labels(directory / "im.csv")
    offset = int(round(ALGORITHM_START_S * fps))
    if offset < 0 or not math.isclose(offset / fps, ALGORITHM_START_S):
        raise ValueError(f"Algorithm start falls outside the frame grid for {animal_id}")
    stop_frame = offset + len(algorithm) * width
    stop_sample = len(algorithm) - trim_samples
    if stop_frame > frame_count or stop_sample <= trim_samples:
        raise ValueError(f"Invalid alignment or trim window for {animal_id}")

    original_traces = {ALGORITHM_SCORER: algorithm}
    reference_times = np.arange(frame_count) / fps
    for expert_id, (times, labels, file_format) in experts.items():
        if len(labels) != frame_count:
            raise ValueError(f"Expert {expert_id} frame count differs from {animal_id}")
        bad_times = np.count_nonzero(np.abs(times - reference_times) > 1e-5)
        if bad_times and (animal_id, expert_id) != KNOWN_TIMESTAMP_EXCEPTION:
            raise ValueError(f"Unexpected timestamps in {animal_id}, E{expert_id}")
        if bad_times:
            print(f"{animal_id}, E{expert_id}: using frame order ({bad_times} irregular time values).")
        if file_format != "CSV":
            print(f"{animal_id}, E{expert_id}: read {file_format} content.")
        original_traces[f"E{expert_id}"] = vote_expert_bins(labels, offset, stop_frame, width)

    timeline_rows.append({
        "animal_id": animal_id, "video_fps": fps, "frame_count": frame_count,
        "algorithm_sampling_hz": SAMPLE_RATE_HZ, "algorithm_start_s": ALGORITHM_START_S,
        "first_valid_frame": offset + trim_samples * width,
        "last_valid_frame": offset + stop_sample * width - 1,
    })
    for scorer, trace in original_traces.items():
        state = reconstruct_states(trace)
        for sample_index, direction in inventory_from_reconstructed(state, trim_samples, stop_sample):
            event_rows.append({
                "animal_id": animal_id, "scorer": scorer, "direction": direction,
                "sample_index": sample_index, "time_s": (offset + sample_index * width) / fps,
            })

video_timeline = pd.DataFrame(timeline_rows)
scorer_events = pd.DataFrame(event_rows, columns=["animal_id", "scorer", "direction", "sample_index", "time_s"])
write_csv(video_timeline, "reconstructed_video_timeline.csv")
print(f"{len(scorer_events)} switches retained after trimming.")
display(video_timeline)

In [ ]:
timeline_by_animal = video_timeline.set_index("animal_id")
animals = video_timeline["animal_id"].tolist()
timeline_view = video_timeline[["animal_id", "video_fps", "algorithm_sampling_hz", "algorithm_start_s"]].copy()
timeline_view["analysis_start_s"] = np.maximum(
    START_TIME_S, video_timeline["first_valid_frame"] / video_timeline["video_fps"]
)
timeline_view["analysis_end_s_exclusive"] = (
    (video_timeline["last_valid_frame"] + 1) / video_timeline["video_fps"]
)
scorer_events = scorer_events.merge(timeline_view, on="animal_id", validate="many_to_one")
expected_time = (
    scorer_events["algorithm_start_s"]
    + scorer_events["sample_index"] / scorer_events["algorithm_sampling_hz"]
)
assert np.allclose(expected_time, scorer_events["time_s"], rtol=0, atol=0.0001), "Event and timeline clocks disagree."

analysis_events = scorer_events.loc[
    scorer_events["time_s"].ge(scorer_events["analysis_start_s"])
    & scorer_events["time_s"].lt(scorer_events["analysis_end_s_exclusive"])
].sort_values(["animal_id", "scorer", "direction", "sample_index"]).reset_index(drop=True)

print(f"{len(scorer_events)} unique scorer transitions; {len(analysis_events)} remain in the analysis interval")
write_csv(analysis_events, "eligible_scoring_events.csv")

display(analysis_events.groupby(["animal_id", "direction", "scorer"]).size()
        .unstack(fill_value=0).reindex(columns=SCORERS, fill_value=0))

## Analysis 1. Timing differences at the onset and end of the same struggle bout

`Sheet1` in `label.xlsx` specifies one observation window per mouse after 120 s in the original video. The window is shared by all scorers. Its boundaries were selected from the red markings on the overview plots and fall within periods that all scorers labeled as immobility before and after the chosen bout. The window includes its start time and excludes its end time.

For the algorithm and each expert (E1–E5), the onset is the first I→S transition in the window. The end is the last subsequent S→I transition, provided the trace remains in I through the end of the window. Each boundary is timestamped at the first sample of the new state in the reconstructed trace, corresponding to the first frame of that time bin. Bout duration is the elapsed time between the two boundaries and includes any intervening periods of immobility. If the onset or a complete return to immobility is missing, the reason is recorded and both boundaries for that scorer are excluded from pairwise comparisons.

For each mouse, the six scorers (algorithm and E1–E5) yield 15 pairs. For each boundary, the time difference is the first scorer’s timestamp minus the second scorer’s timestamp. A negative value means the first scorer marked the transition earlier.

In [ ]:
first_bout_windows = pd.read_excel(
    RAW_DATA_DIR / "label.xlsx",
    sheet_name="Sheet1",
    usecols=["animal_id", "window_start_s", "window_end_s"],
).set_index("animal_id", verify_integrity=True)
if set(first_bout_windows.index) != set(animals):
    raise ValueError("label.xlsx must list exactly one window per animal.")
first_bout_windows = first_bout_windows.loc[animals].apply(pd.to_numeric, errors="raise")
lower = first_bout_windows["window_start_s"]
upper = first_bout_windows["window_end_s"]
analysis_bounds = timeline_view.set_index("animal_id").loc[animals]
valid_windows = (
    np.isfinite(lower) & np.isfinite(upper)
    & lower.ge(analysis_bounds["analysis_start_s"])
    & upper.le(analysis_bounds["analysis_end_s_exclusive"])
    & lower.lt(upper)
)
if not valid_windows.all():
    raise ValueError(f"Invalid first bout windows in label.xlsx: {lower.index[~valid_windows].tolist()}")
display(first_bout_windows)


def first_struggle_bout(events, window_start_s, window_end_s):
    in_window = events["time_s"].ge(window_start_s) & events["time_s"].lt(window_end_s)
    ordered = events.loc[in_window].sort_values("sample_index")
    starts = ordered.loc[ordered["direction"].eq("I_to_S")]
    if starts.empty:
        return np.nan, np.nan, "no_start_in_window"
    start = starts.iloc[0]
    end = ordered.iloc[-1]
    # The final transition must leave the trace immobile through the end of the window.
    if end["direction"] != "S_to_I":
        return start["time_s"], np.nan, "no_sustained_return_in_window"
    return start["time_s"], end["time_s"], "complete"


event_groups = analysis_events.groupby(["animal_id", "scorer"])
empty_events = analysis_events.iloc[0:0]
bout_rows = []
for animal_id, scorer in product(animals, SCORERS):
    key = animal_id, scorer
    events = event_groups.get_group(key) if key in event_groups.groups else empty_events
    window_start_s, window_end_s = first_bout_windows.loc[animal_id]
    start_s, end_s, status = first_struggle_bout(events, window_start_s, window_end_s)
    bout_rows.append({
        "animal_id": animal_id,
        "scorer": scorer,
        "I_to_S": start_s,
        "S_to_I": end_s,
        "struggle_duration_s": end_s - start_s,
        "status": status,
        "window_start_s": window_start_s,
        "window_end_s": window_end_s,
    })
first_bouts = pd.DataFrame(bout_rows)
complete = first_bouts["status"].eq("complete")
assert (first_bouts.loc[complete, "struggle_duration_s"] > 0).all()
print(f"Complete first struggle bouts: {complete.sum()}/{len(first_bouts)}")
write_csv(first_bouts, "first_struggle_bouts.csv")
display(first_bouts.round(3))

eligible_edges = first_bouts[["animal_id", "scorer", *DIRECTIONS]].copy()
eligible_edges.loc[~complete, DIRECTIONS] = np.nan
first_bout_edges = (
    eligible_edges.melt(id_vars=["animal_id", "scorer"], var_name="direction", value_name="time_s")
    .pivot(index=["animal_id", "direction"], columns="scorer", values="time_s")
    .reindex(pd.MultiIndex.from_product([animals, DIRECTIONS], names=["animal_id", "direction"]))
    .reindex(columns=SCORERS)
)

scorer_pairs = list(combinations(SCORERS, 2))
assert len(scorer_pairs) == 15
delta_rows = []
for (animal_id, direction), row in first_bout_edges.iterrows():
    for first, second in scorer_pairs:
        delta_rows.append({
            "animal_id": animal_id,
            "direction": direction,
            "comparison": f"{first} – {second}",
            "comparison_type": "Algorithm vs expert" if first == ALGORITHM_SCORER else "Expert vs expert",
            "first_scorer": first,
            "second_scorer": second,
            "first_scorer_time_s": row[first],
            "second_scorer_time_s": row[second],
            "delta_s": row[first] - row[second],
        })
first_differences = pd.DataFrame(delta_rows)
first_differences["abs_delta_s"] = first_differences["delta_s"].abs()
write_csv(first_differences, "first_bout_pairwise_differences.csv")
display(first_differences.head(8).round(3))
print(f"Pairwise rows: {len(first_differences)}; missing because a bout is incomplete: "
      f"{first_differences['delta_s'].isna().sum()}")

In [ ]:
difference_summary = first_differences.groupby(["direction", "comparison_type"], sort=False).agg(
    available_pairs=("delta_s", "count"),
    signed_median_s=("delta_s", "median"),
    signed_min_s=("delta_s", "min"),
    signed_max_s=("delta_s", "max"),
    absolute_median_s=("abs_delta_s", "median"),
    absolute_q1_s=("abs_delta_s", lambda values: values.quantile(0.25)),
    absolute_q3_s=("abs_delta_s", lambda values: values.quantile(0.75)),
)
write_csv(difference_summary, "first_bout_difference_summary.csv", index=True)
display(difference_summary.round(3))

pair_labels = [f"{first} – {second}" for first, second in scorer_pairs]
pair_x = {label: x for x, label in enumerate(pair_labels)}
animal_offsets = np.linspace(-0.28, 0.28, len(animals)) if len(animals) > 1 else [0.0]
animal_colors = {animal_id: ANIMAL_COLORS[index % len(ANIMAL_COLORS)]
                 for index, animal_id in enumerate(animals)}
pair_colors = {
    label: "#16777a" if first == ALGORITHM_SCORER else "#b5bdc5"
    for label, (first, _) in zip(pair_labels, scorer_pairs)
}
fig, axes = plt.subplots(1, 2, figsize=(15, 6.2))
fig.subplots_adjust(left=0.07, right=0.985, bottom=0.31, top=0.88, wspace=0.22)
fig.text(0.5, 0.965, "Bands show mean ± SEM. Black marks show the median.",
         ha="center", va="top", fontsize=10)
for direction, ax in zip(DIRECTIONS, axes):
    direction_data = first_differences.loc[first_differences["direction"].eq(direction)]
    distribution = (
        direction_data.dropna(subset=["delta_s"]).groupby("comparison", sort=False)["delta_s"]
        .agg(mean="mean", sem="sem", median="median")
    )
    bands = distribution.dropna(subset=["sem"])
    ax.bar(
        bands.index.map(pair_x), 2 * bands["sem"], bottom=bands["mean"] - bands["sem"],
        width=0.74, color=bands.index.map(pair_colors), edgecolor="none", zorder=2,
    )
    positions = distribution.index.map(pair_x).to_numpy()
    ax.hlines(distribution["median"], positions - 0.32, positions + 0.32,
              colors="#202427", linewidth=1.3, zorder=3)
    for animal_id, offset in zip(animals, animal_offsets):
        points = direction_data.loc[direction_data["animal_id"].eq(animal_id)].dropna(subset=["delta_s"])
        ax.scatter(
            points["comparison"].map(pair_x) + offset, points["delta_s"],
            label=animal_id, s=26, marker="o", facecolors=animal_colors[animal_id],
            edgecolors="white", linewidths=0.55, zorder=4,
        )
    ax.axhline(0, color="#a3a9af", linewidth=0.9, zorder=1)
    ax.set(title=DIRECTION_LABELS[direction], ylabel="First complete bout boundary difference (s)")
    ax.set_xticks(range(len(pair_labels)), pair_labels, rotation=58, ha="right")
    ax.grid(axis="y", alpha=0.2)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, 0.04),
           ncol=min(6, len(labels)), fontsize=9, frameon=False)
save_figure(fig, "first_bout_boundary_differences.pdf")

## Analysis 2. Algorithm–expert match rates within ±2 s by mouse and transition direction

Each algorithm transition is counted once per expert comparison. For each mouse and direction, the algorithm is paired separately with E1–E5. Matching uses integer sample indices. Two events are eligible if their timestamps differ by at most 2 s. Matching is one-to-one within each expert comparison. The procedure first maximizes the number of matches, then minimizes the sum of absolute timing differences among the matched pairs.

Each matrix cell gives the proportion of algorithm transitions matched by that expert. For a given mouse and direction, all five expert rates use the same set of algorithm transitions as their denominator. Events marked only by an expert are excluded from the denominator. If the algorithm recorded no transitions for that mouse and direction, the rate is reported as NA.

In [ ]:
def match_transitions(algorithm_samples, expert_samples, tolerance_samples):
    """Return one-to-one pairs ordered by time within an inclusive tolerance."""
    n_algorithm, n_expert = len(algorithm_samples), len(expert_samples)
    # Score by (number of hits, negative total timing cost), compared lexicographically.
    scores = [[(0, 0)] * (n_expert + 1) for _ in range(n_algorithm + 1)]
    moves = [[None] * (n_expert + 1) for _ in range(n_algorithm + 1)]
    for i in range(1, n_algorithm + 1):
        for j in range(1, n_expert + 1):
            candidates = [(scores[i - 1][j], "algorithm"), (scores[i][j - 1], "expert")]
            distance = abs(algorithm_samples[i - 1] - expert_samples[j - 1])
            if distance <= tolerance_samples:
                hits, signed_cost = scores[i - 1][j - 1]
                candidates.append(((hits + 1, signed_cost - distance), "match"))
            score, move = max(candidates, key=lambda candidate: candidate[0])
            scores[i][j], moves[i][j] = score, move

    pairs = []
    i, j = n_algorithm, n_expert
    while i and j:
        move = moves[i][j]
        if move == "match":
            pairs.append((algorithm_samples[i - 1], expert_samples[j - 1]))
            i, j = i - 1, j - 1
        elif move == "algorithm":
            i -= 1
        else:
            j -= 1
    return pairs[::-1]


event_index = (
    analysis_events.groupby(["animal_id", "direction", "scorer"])["sample_index"]
    .agg(lambda samples: sorted(samples.tolist())).to_dict()
)
event_rows = []
for animal_id, direction, scorer in product(animals, DIRECTIONS, SCORERS[1:]):
    hz = timeline_by_animal.loc[animal_id, "algorithm_sampling_hz"]
    origin = timeline_by_animal.loc[animal_id, "algorithm_start_s"]
    algorithm_samples = event_index.get((animal_id, direction, ALGORITHM_SCORER), [])
    expert_samples = event_index.get((animal_id, direction, scorer), [])
    tolerance_samples = int(np.floor(np.nextafter(MATCH_TOLERANCE_S * hz, np.inf)))
    matches = match_transitions(algorithm_samples, expert_samples, tolerance_samples)
    assert len({a for a, _ in matches}) == len({e for _, e in matches}) == len(matches)
    assert all(abs(a - e) <= tolerance_samples for a, e in matches)
    expert_by_algorithm = dict(matches)
    for algorithm_sample in algorithm_samples:
        expert_sample = expert_by_algorithm.get(algorithm_sample)
        event_rows.append({
            "animal_id": animal_id,
            "direction": direction,
            "scorer": scorer,
            "algorithm_time_s": origin + algorithm_sample / hz,
            "expert_time_s": origin + expert_sample / hz if expert_sample is not None else np.nan,
            "delta_s": (algorithm_sample - expert_sample) / hz if expert_sample is not None else np.nan,
            "matched": expert_sample is not None,
        })

columns = ["animal_id", "direction", "scorer", "algorithm_time_s", "expert_time_s", "delta_s", "matched"]
algorithm_event_matches = pd.DataFrame(event_rows, columns=columns)
comparison_grid = pd.MultiIndex.from_product(
    [animals, DIRECTIONS, SCORERS[1:]], names=["animal_id", "direction", "scorer"]
)
expert_match_rates = (
    algorithm_event_matches.groupby(["animal_id", "direction", "scorer"])["matched"].mean()
    .reindex(comparison_grid).mul(100).rename("match_rate_pct")
)
matrix_group = expert_match_rates.groupby(level=["animal_id", "direction"])
row_maxima = matrix_group.transform("max")
match_rate_table = expert_match_rates.to_frame()
match_rate_table["is_row_max"] = expert_match_rates.notna() & expert_match_rates.eq(row_maxima)
write_csv(algorithm_event_matches, "algorithm_event_matches.csv")
write_csv(match_rate_table, "expert_match_rates.csv", index=True)
print("Event-level audit for individual algorithm transitions")
display(algorithm_event_matches.head(10).round(3))
print("Match rate (%) by animal, direction and expert")
display(expert_match_rates.unstack("scorer").round(1))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.4, 4.9), sharey=True, constrained_layout=True)
for direction, ax in zip(DIRECTIONS, axes):
    panel = expert_match_rates.xs(direction, level="direction").unstack("scorer")
    panel = panel.reindex(index=animals, columns=SCORERS[1:])
    top = match_rate_table["is_row_max"].xs(direction, level="direction").unstack("scorer")
    top = top.reindex(index=animals, columns=SCORERS[1:]).to_numpy()
    write_csv(panel, f"expert_match_rate_matrix_{direction}.csv", index=True)
    values = panel.to_numpy()
    heatmap = ax.pcolormesh(
        np.arange(values.shape[1] + 1) - 0.5, np.arange(values.shape[0] + 1) - 0.5,
        np.ma.masked_invalid(values), vmin=0, vmax=100, cmap=BLUE_RED_CMAP,
        shading="flat", linewidth=0.6, edgecolors="white", rasterized=False,
    )
    for y, x in product(range(len(animals)), range(len(SCORERS) - 1)):
        value = values[y, x]
        ax.text(x, y, f"{value:.1f}%" if np.isfinite(value) else "NA",
                ha="center", va="center", color="black",
                fontweight="bold" if top[y, x] else "normal")
    ax.set_xticks(range(len(SCORERS) - 1), SCORERS[1:])
    ax.set_yticks(range(len(animals)), animals)
    ax.set(title=DIRECTION_LABELS[direction], xlabel="Expert compared with algorithm")
    ax.tick_params(length=0)
axes[0].invert_yaxis()
colorbar = fig.colorbar(heatmap, ax=axes, label="Matched algorithm events (%)", shrink=0.8)
colorbar.solids.set_rasterized(False)
save_figure(fig, "expert_match_rate_heatmaps.pdf")

In [ ]:
# One aligned overview per mouse/session on the same time grid as the analyses.
overview_dir = ANALYSIS_DIR / "animal_overviews"
overview_dir.mkdir(exist_ok=True)


def read_area_change(path, algorithm_length):
    with path.open(encoding="utf-8-sig", newline="") as stream:
        header, row = list(csv.reader(stream))
    values = np.asarray(row, dtype=float)
    if len(header) != values.size or values.size != algorithm_length + 1 or not np.isfinite(values).all():
        raise ValueError(f"Unexpected number or type of samples in {path}")
    return values


def plot_animal_overview(timeline):
    animal_id = timeline.animal_id
    folder = RAW_DATA_DIR / "behavioral_scoring_validation video" / animal_id
    fps = timeline.video_fps
    width = fps // timeline.algorithm_sampling_hz
    offset = round(timeline.algorithm_start_s * fps)
    display_start = max(START_TIME_S, timeline.first_valid_frame / fps)
    display_end = (timeline.last_valid_frame + 1) / fps
    start = math.floor((display_start * fps - offset) / width)
    stop = (timeline.last_valid_frame + 1 - offset) // width
    sample_edges_s = (offset + np.arange(start, stop + 1) * width) / fps

    algorithm = read_algorithm_labels(folder / "im.csv")
    raw_area = read_area_change(folder / "Area_Change_raw.csv", len(algorithm))[start:stop]
    area = read_area_change(folder / "Area_Change.csv", len(algorithm))[start:stop]
    state_tracks = [(ALGORITHM_SCORER, reconstruct_states(algorithm)[start:stop], "#235d86")]
    expert_stem = EXPERT_ALIASES.get(animal_id, animal_id)
    for expert_id in EXPERT_IDS:
        _, labels, _ = read_original_expert(RAW_DATA_DIR / f"expert {expert_id}" / f"{expert_stem}.csv")
        if len(labels) != timeline.frame_count:
            raise ValueError(f"Frame count mismatch for {animal_id}, expert {expert_id}")
        voted = vote_expert_bins(labels, offset, offset + len(algorithm) * width, width)
        state_tracks.append((f"Expert {expert_id}", reconstruct_states(voted)[start:stop], "#524d55"))

    fig, axes = plt.subplots(
        8, 1, sharex=True, figsize=(15.5, 9.4),
        gridspec_kw={"height_ratios": [1.45, 1.45, 1, 1, 1, 1, 1, 1]},
    )
    fig.subplots_adjust(left=0.16, right=0.985, top=0.86, bottom=0.095, hspace=0.30)
    fig.text(0.027, 0.96, f"{animal_id}   |   Area change and behavioral states",
             fontsize=20, fontweight="bold", va="top")
    fig.text(0.028, 0.907,
             f"Assumed algorithm origin {timeline.algorithm_start_s:g} s   |   I = immobility, S = struggle"
             f"   |   {TRIM_SECONDS:g} s trimmed from each end of the common window",
             fontsize=10.5, va="top")

    low = min(0.0, raw_area.min(), area.min())
    high = max(raw_area.max(), area.max())
    for ax, label, values, color in (
        (axes[0], "Area_Change_Raw", raw_area, "#55717e"),
        (axes[1], "Area_Change", area, "#286c82"),
    ):
        ax.plot(sample_edges_s[:-1], values, color=color, linewidth=0.8)
        ax.set_ylim(low, high + max(0.04 * (high - low), 0.01))
        ax.text(-0.061, 0.5, label, transform=ax.transAxes, ha="right", va="center", fontsize=10)

    for ax, (label, states, color) in zip(axes[2:], state_tracks, strict=True):
        ax.stairs(states, sample_edges_s, baseline=None, color=color, linewidth=1.15)
        ax.set_ylim(-0.16, 1.16)
        ax.set_yticks([0, 1], ["I", "S"])
        ax.text(-0.061, 0.5, label, transform=ax.transAxes, ha="right", va="center", fontsize=10)

    window_start_s, window_end_s = first_bout_windows.loc[animal_id]
    for ax in axes:
        ax.axvspan(window_start_s, window_end_s, facecolor="#fff2bf", edgecolor="none", zorder=0)
        ax.spines[["left", "right", "top"]].set_visible(False)
        ax.spines["bottom"].set_color("#cbd5dc")
        ax.set_axisbelow(True)
        ax.grid(axis="x", color="#d2dce2", linewidth=0.65)
        ax.tick_params(axis="both", length=0, labelsize=9)

    axes[-1].set_xticks(np.arange(math.ceil(display_start / 30) * 30, display_end, 30))
    axes[-1].set_xlim(display_start, display_end)
    axes[-1].set_xlabel(
        f"Video time (s)   |   Analysis window ({display_start:g}-{display_end:g} s)"
        f"   |   Shading: first bout window ({window_start_s:g}-{window_end_s:g} s)"
    )
    filename = f"animal_overviews/{animal_id}_overview.pdf"
    save_figure(fig, filename)  # PDF vectors, FIGURE_DPI = 300


for timeline_row in video_timeline.itertuples(index=False):
    plot_animal_overview(timeline_row)